In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/FMCG_Project/consolidated_pipeline/1_setup/utilities  

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'products', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = "/Volumes/fmcg/sportsbar-dp/products/*.csv"

In [0]:
df = spark.read.csv(base_path, 
                    header=True, 
                    inferSchema=True
                    )
df = df.withColumn('read_timestamp', F.current_timestamp()) \
       .select("*", "_metadata.file_name", "_metadata.file_size")

display(df.limit(10))

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-04-10T11:39:28.586Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-04-10T11:39:28.586Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-04-10T11:39:28.586Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-04-10T11:39:28.586Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-04-10T11:39:28.586Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-04-10T11:39:28.586Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-04-10T11:39:28.586Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-04-10T11:39:28.586Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-04-10T11:39:28.586Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-04-10T11:39:28.586Z,products.csv,1388


In [0]:
df.write.format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
display(spark.sql("select * from fmcg.bronze.products"))

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-04-10T11:45:31.531Z,products.csv,1388


#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-04-10T11:45:31.531Z,products.csv,1388


##### Transformations

1. Drop Duplicates

In [0]:
print("Rows before duplicates dropped:", df_bronze.count())
df_silver = df_bronze.dropDuplicates(['product_id'])
print("Rows after duplicates dropped:", df_silver.count())

Rows before duplicates dropped: 20
Rows after duplicates dropped: 18


2. Case Fix

In [0]:
df_silver = df_silver.withColumn("category", 
                                 F.when(F.col("category").isNull(), None).otherwise(F.initcap("category"))
)
df_silver.select("category").distinct().show()

+-----------------+
|         category|
+-----------------+
|      Energy Bars|
|     Protien Bars|
|Granola & Cereals|
|   Recovery Dairy|
|   Healthy Snacks|
|  Electrolyte Mix|
+-----------------+



3. Fix spelling mistake in Protein

In [0]:
df_silver = df_silver.withColumn("product_name", 
                                 F.regexp_replace(F.col("product_name"), "(?i)protien", "Protein")).withColumn("category", 
                                 F.regexp_replace(F.col("category"), "(?i)protien", "Protein"))
display(df_silver)

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,Energy Bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,Energy Bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Protein Bar Peanut Crunch (45g),25891201,Protein Bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protein Bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Protein Bar Peanut Crunch (65g),25891203,Protein Bars,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-04-10T11:45:31.531Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,Recovery Dairy,2026-04-10T11:45:31.531Z,products.csv,1388


##### Standardizing customer attributes to match parent company data model

In [0]:
# add division column
df_silver = df_silver.withColumn("division", 
                                 F.when(F.col("category") == "Energy Bars", "Nutrition Bars")
                                  .when(F.col("category") == "Protein Bars", "Nutrition Bars")
                                  .when(F.col("category") == "Granola & Cereals", "Breakfast Foods")
                                  .when(F.col("category") == "Recovery Dairy", "Dairy & Recovery")
                                  .when(F.col("category") == "Healthy Snacks", "Healthy Snacks")
                                  .when(F.col("category") == "Electrolyte Mix", "Hydration & Electrolytes")
                                  .otherwise("other")
                                 )

# add variant column
df_silver = df_silver.withColumn("variant",
                                 F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
                                 )
                
display(df_silver)

product_name,product_id,category,read_timestamp,file_name,file_size,division,variant
SportsBar Energy Bar Choco Fudge (60g),25891101,Energy Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,60g
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,40g
SportsBar Energy Bar Choco Fudge (25g),25891103,Energy Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,25g
SportsBar Protein Bar Peanut Crunch (45g),25891201,Protein Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,45g
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protein Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,55g
SportsBar Protein Bar Peanut Crunch (65g),25891203,Protein Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,65g
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-04-10T11:45:31.531Z,products.csv,1388,Breakfast Foods,400g
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-04-10T11:45:31.531Z,products.csv,1388,Breakfast Foods,300g
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-04-10T11:45:31.531Z,products.csv,1388,Breakfast Foods,200g
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,Recovery Dairy,2026-04-10T11:45:31.531Z,products.csv,1388,Dairy & Recovery,200g


3: Create new column: product_code


In [0]:
# Invalid product_ids are replaced with a fallback value to avoid losing fact records and ensure downstream joins remain consistent

# Generate deterministic product_code from product_name
# Clean product ids. Keep only numeric ids. Else set to 999999

df_silver = df_silver.withColumn(
    "product_code",
    F.sha2(F.col("product_name").cast("string"), 256)
).withColumn(
    "product_id",
    F.when(
        F.col("product_id").cast("string").rlike("^[0-9]+$"),
        F.col("product_id").cast("string")
    ).otherwise(F.lit(999999).cast("string"))
).withColumnRenamed(
    "product_name", "product"
)

display(df_silver)

product,product_id,category,read_timestamp,file_name,file_size,division,variant,product_code
SportsBar Energy Bar Choco Fudge (60g),25891101,Energy Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,60g,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,40g,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
SportsBar Energy Bar Choco Fudge (25g),25891103,Energy Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,25g,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268
SportsBar Protein Bar Peanut Crunch (45g),25891201,Protein Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,45g,2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protein Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,55g,0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28
SportsBar Protein Bar Peanut Crunch (65g),25891203,Protein Bars,2026-04-10T11:45:31.531Z,products.csv,1388,Nutrition Bars,65g,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-04-10T11:45:31.531Z,products.csv,1388,Breakfast Foods,400g,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-04-10T11:45:31.531Z,products.csv,1388,Breakfast Foods,300g,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-04-10T11:45:31.531Z,products.csv,1388,Breakfast Foods,200g,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,Recovery Dairy,2026-04-10T11:45:31.531Z,products.csv,1388,Dairy & Recovery,200g,da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05


In [0]:
df_silver = df_silver.select('product_code', 'division', 'category', 'product', 'variant', 'product_id', 'read_timestamp', 'file_name', 'file_size')
display(df_silver)

product_code,division,category,product,variant,product_id,read_timestamp,file_name,file_size
e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (60g),60g,25891101,2026-04-10T11:45:31.531Z,products.csv,1388
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g,25891102,2026-04-10T11:45:31.531Z,products.csv,1388
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (25g),25g,25891103,2026-04-10T11:45:31.531Z,products.csv,1388
2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (45g),45g,25891201,2026-04-10T11:45:31.531Z,products.csv,1388
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),55g,25891202,2026-04-10T11:45:31.531Z,products.csv,1388
889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (65g),65g,25891203,2026-04-10T11:45:31.531Z,products.csv,1388
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g,25891301,2026-04-10T11:45:31.531Z,products.csv,1388
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (300g),300g,25891302,2026-04-10T11:45:31.531Z,products.csv,1388
c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (200g),200g,25891303,2026-04-10T11:45:31.531Z,products.csv,1388
da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (200g),200g,25891401,2026-04-10T11:45:31.531Z,products.csv,1388


Write to silver schema

In [0]:
df_silver.write.format('delta').option("delta.enableChangeDataFeed", "true").option("mergeSchema", "true").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

##### Gold Processing

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")

# select required columns only for gold layer
df_gold = df_silver.select('product_code', 'division', 'category', 'product', 'variant')
display(df_gold)

product_code,division,category,product,variant
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),180g
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (300g),300g
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (80g),80g
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g
5931334e4cbe6b3792c209e8394e87aa21b83816b47d99375e4ff25e651ce63a,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (350g),350g
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),55g
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,Hydration & Electrolytes,Electrolyte Mix,SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),30 Sachets
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,Hydration & Electrolytes,Electrolyte Mix,SportsBar Electrolyte Mix Lemon-Lime (15 Sachets),15 Sachets
c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (200g),200g


In [0]:
df_gold.write\
    .format('delta')\
        .option("delta.enableChangeDataFeed", "True")\
            .mode("overwrite")\
                .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
df_child = spark.sql(f"select * from {catalog}.{gold_schema}.sb_dim_{data_source}")
display(df_child)
df_child.createOrReplaceTempView('df_child')

In [0]:
spark.sql(f"""
          MERGE INTO fmcg.gold.dim_products target
          USING df_child SOURCE
          ON target.product_code = source.product_code
          WHEN MATCHED THEN
          UPDATE SET *
          WHEN NOT MATCHED THEN
          INSERT *
          """)

print("Merge completed successfully")

Merge completed successfully


In [0]:
display(spark.sql("select * from fmcg.gold.dim_products"))

product_code,division,category,product,variant
ARCHDDE20D,Archery,Arrows,PX Carbon Arrow Set,12 Pack
ARCH158F41,Archery,Finger Tab,LX Leather Finger Tab,Universal
ARCHAFF0E4,Archery,Bow,AX Precision Recurve Bow,26 lbs
ARCH6B94F7,Archery,Bow,AX Precision Recurve Bow,30 lbs
ARCH5D1FE7,Archery,Bow Stringer,BX Bow Stringing Tool,Standard
ARCH7B49A9,Archery,Arm Guard,NX Archery Arm Guard,Medium
ARCH497D34,Archery,Target,TX Foam Archery Target,80 cm
ARCHE71D79,Archery,Arm Guard,NX Archery Arm Guard,Large
ARCHDD8749,Archery,Target,TX Foam Archery Target,60 cm
BADMC045D4,Badminton,Badminton Racket,BX AeroLite Badminton Racket,Head Heavy
